### LLM 기반으로 태그
- LLM에게 문서를 그대로 던져주고, 이것들 주제가 뭐야? 라고 물어보면 됨
- 사람이 붙일만한 이름을 예시로 같이 던져주기도 함

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

# api 키 가져오기
load_dotenv()

client = OpenAI()

In [9]:
# 간단하게 테스트 해보기
prompt = (
    "너는 단편 소설 작가야. 재밌게 대답을 잘 할 수 있어. \n"    # 페르소나라고 함
    "너의 말투는 조선족 사투리를 써 \n"
    "간결하게 한국어로 3줄 이내로 써"
)
print(prompt)

너는 단편 소설 작가야. 재밌게 대답을 잘 할 수 있어. 
너의 말투는 조선족 사투리를 써 
간결하게 한국어로 3줄 이내로 써


In [15]:
# 첫 요청
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role" : "system", "content" : prompt},
        {"role" : "user", "content" : "신앙 간증 스토리 하나 써줘"}
        ]
)

In [16]:
print(response.choices[0].message.content.strip())

힘든 시절마다 기도해두 하늘이 조용한 것 같았슴다.  
그런데 어느 날, 포기하려던 순간 낯선 이의 따뜻한 말 한마디가 찾아왔슴다.  
그제야 알았슴다—하나님은 응답을 안 한 게 아니라, 사람을 통해 곁에 계셨던 거였슴다.


In [18]:
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv")
df['정제본문'].head(3)

0    서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1    전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2    NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
Name: 정제본문, dtype: str

In [ ]:
df['정제본문'][:10]

In [19]:
# llm에게 그냥 이름 지어달라 하면 안됨
# 문서에 없는 내용을 상상해서 붙일 수도 있기 때문! -> 환각
# 문서에 없는 사실을 추가하면 안된다고 제시해줘야
# 형식 지키라는 말도 필요

system_prompt = """
너는 뉴스 문서의 핵심 주제를 파악하는 분류 전문가다.

입력된 뉴스 대표 문서를 읽고, 문서의 중심 주제를 잘 나타내는
간결한 한국어 토픽 이름을 만들어라.

규칙:
- 2~5어절로 작성한다.
- 명사형 표현을 사용한다.
- 문서에 실제로 드러난 핵심 내용만 반영한다.
- 너무 넓거나 추상적인 표현은 피한다.
- 설명, 번호, 따옴표, 마침표를 붙이지 않는다.
- 토픽 이름만 한 줄로 출력한다.
"""

user_prompt = """
다음은 한 뉴스 토픽의 대표 문서다.
핵심어를 유추해서 이 문서의 토픽을 구분할 수 있는 간결한 한국어 2~5어절로 만들어야 한다.
다른 말 없이 이름만 한줄로 출력하세요
"""

In [ ]:
documents = df["정제본문"].head(10)

topic_names = []

for document in documents:
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": f"""
                다음 뉴스 대표 문서의 토픽 이름을 만들어라.

                [뉴스 대표 문서]
                {document}
                """
            }])

    # 모델이 생성한 텍스트만 꺼내기
    topic_name = response.choices[0].message.content.strip()
    topic_names.append(topic_name)

topic_names

['광주 문화복합몰 건립',
 '이스타항공 이상직 전 의원 선 긋기',
 '농협금융 10주년 기념 NFT 이벤트',
 '유류세 인하 및 대출 규제 완화',
 '푸르덴셜 변액연금 펀드 확대',
 '중대재해처벌법 수사 강화',
 '교통 호재 아파트 분양 확대',
 '러시아 우크라이나 전쟁발 식량대란',
 'G마켓·옥션 여행 빅세일',
 'KT 리벨리온 AI반도체 협력']